# 🅿️ Notebook 1: Parking Lot — Class Design

Classic **object-oriented design (OOD)** interview question. In real interviews you are judged
less on "did your code run" and more on *how you reason about entities, responsibilities,
and extension points*. So before writing any code we:

1. Clarify requirements (ask clarifying questions).
2. Identify actors and use-cases.
3. Identify the nouns (classes) and verbs (methods).
4. Draw the class relationships.
5. Sanity-check against the **SOLID** principles.

Notebook 2 then implements it with a clear **bad → good → best** progression, and
notebook 3 extends it with real-world concerns (concurrency, reserved spots, payments).


## 🛠️ Setup

```bash
cd 07-object-oriented-design/parking-lot
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear,
reload the window: `Cmd+Shift+P` → **Reload Window**.


## 1. Clarifying questions (what to ask the interviewer)

In a real interview, *asking questions is part of the grade*. Example questions:

- How many **levels / floors**? How many **spots** per level?
- What **vehicle types** do we support? (motorcycle, car, truck/van, bus, EV…)
- Are there **reserved** spots? (handicapped, EV-charging, compact…)
- How is pricing computed? (flat fee, per-hour, tiered, monthly pass?)
- Do we accept **cash, card, app**? Multiple payment methods?
- Is it **one entry/exit** or many? (matters for concurrency)
- Do we need to **track** license plates, display availability on a sign, etc.?

> 💡 *Tip:* write your assumptions down. Scope is a design decision, not a failure to deliver.

### Our assumptions for this lab

- Multiple **levels**, each with many **spots**.
- Three vehicle sizes: **motorcycle**, **car**, **truck**.
- A spot fits any vehicle **≤ its size** (a truck spot fits a bike; a car spot does *not* fit a truck).
- On entry we issue a **ticket**. On exit we release the spot and **charge by time**.
- Pricing rate depends on vehicle size (big vehicles pay more).


## 2. Actors and use-cases

An **actor** is anyone (or anything) that interacts with the system.

| Actor         | What they do                                |
|---------------|----------------------------------------------|
| Driver        | enters, takes a ticket, parks, pays, leaves |
| Attendant     | handles payment, special cases              |
| System admin  | adds/removes spots, sets pricing            |

### Core use-cases (happy path)

1. `park(vehicle)` → returns a **Ticket** or fails if the lot is full.
2. `leave(ticket)` → frees the spot and returns the **fee**.
3. `available_spots()` → reports free capacity (for the big display at the entrance).


## 3. Finding the classes (nouns) and methods (verbs)

Underline the nouns in the requirements; they're candidate classes:

> *The parking **lot** has **levels**. Each level has **spots** of different **sizes**.
> A **vehicle** arrives and gets a **ticket**. On exit the **ticket** is used to compute a **fee**.*

Candidate classes:

- `ParkingLot`   — the whole thing.
- `Level`        — one floor.
- `Spot`         — one parking space.
- `Vehicle`      — abstract, with subclasses: `Motorcycle`, `Car`, `Truck`.
- `Ticket`       — the slip you get at entry.
- `Size`         — an enum of sizes (MOTORCYCLE, CAR, TRUCK).

Verbs become methods: `park`, `leave`, `can_fit`, `find_spot`, `compute_fee`.


## 4. UML-style class diagram (ASCII)

```
┌──────────────┐ 1   *  ┌─────────┐ 1   *  ┌────────┐
│ ParkingLot   │───────▶│  Level  │───────▶│  Spot  │
└──────────────┘        └─────────┘        └────────┘
      │                                        │
      │ issues                               parks
      ▼                                        ▼
┌──────────────┐                         ┌────────────┐
│   Ticket     │◀──────references────────│  Vehicle   │
└──────────────┘                         └────────────┘
                                               △
                                               │ (inherits)
                               ┌───────────────┼───────────────┐
                               │               │               │
                          Motorcycle          Car            Truck
```

### Responsibilities (one-liners)

- **`Vehicle`** *(abstract)*: knows its own `size`.
- **`Spot`**: knows its own `size`; can `park(vehicle)` / `leave()`; `can_fit()`.
- **`Level`**: owns spots; `find_spot(vehicle)`.
- **`ParkingLot`**: owns levels; `park(vehicle) -> Ticket`; `leave(ticket) -> fee`.
- **`Ticket`**: immutable record tying *vehicle* + *spot* + *entry_time*.


## 5. SOLID sanity check

SOLID is 5 rules of thumb for clean OOD. Beginner-friendly version:

| Letter | Rule (plain English) | How our design respects it |
|--------|----------------------|-----------------------------|
| **S** — Single responsibility | One class does one thing. | `Spot` only tracks occupancy. `Ticket` only holds data. |
| **O** — Open/closed | Open for extension, closed for modification. | Add `Bus` as a new `Vehicle` subclass without editing `Spot`. |
| **L** — Liskov substitution | Subclasses must be usable wherever the parent is. | Any `Vehicle` subclass works anywhere a `Vehicle` is expected. |
| **I** — Interface segregation | Small, focused interfaces. | `Spot` exposes just `can_fit / park / leave`, not payment. |
| **D** — Dependency inversion | Depend on abstractions, not concretions. | `ParkingLot` uses a `PricingStrategy` (notebook 2), not a hard-coded table. |

> 🎯 If you can't point at *where* each SOLID principle lives in your design, the
> interviewer likely will.


## 6. A tiny runnable sanity check

Let's make sure the `Size` enum encodes the fit-rule correctly.

In [ ]:
from enum import Enum

class Size(Enum):
    MOTORCYCLE = 1
    CAR        = 2
    TRUCK      = 3

def fits(vehicle_size: Size, spot_size: Size) -> bool:
    # A spot fits any vehicle whose size <= spot's size.
    return vehicle_size.value <= spot_size.value

# A few examples:
print('bike in car spot? ', fits(Size.MOTORCYCLE, Size.CAR))    # True
print('truck in car spot?', fits(Size.TRUCK,      Size.CAR))    # False
print('car in truck spot?', fits(Size.CAR,        Size.TRUCK))  # True


👉 **Next:** Notebook 2 walks through **bad → good → best** implementations so you
can *feel* why the OOD principles matter.